# 00 — Setup

**Project:** Robust and Explainable AI for Diabetic Retinopathy.

Run this notebook **once per Colab session** before any phase notebook. It:
1. Mounts Google Drive
2. Installs all dependencies
3. Extracts `APTOS.zip` and `EYEQ.zip` from `/content/drive/MyDrive/Thesis/` to local Colab disk
4. Creates the per-phase results directory tree on Drive (so outputs persist across sessions)

> **Runtime:** GPU recommended (T4 / A100). `Runtime → Change runtime type → GPU`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

In [ ]:
%%capture
!pip install -q timm captum shap grad-cam scikit-image opencv-python-headless tabulate
!pip install -q diffusers transformers accelerate

## 3. Project paths (single source of truth)

These paths are duplicated at the top of every phase notebook so each notebook is self-contained. Edit here first, then mirror into the phase notebooks if you change anything.

In [ ]:
from pathlib import Path

# ----- Drive (input + persistent output) -----
DRIVE_ROOT      = Path('/content/drive/MyDrive/Thesis')
APTOS_ZIP       = DRIVE_ROOT / 'APTOS.zip'
EYEQ_ZIP        = DRIVE_ROOT / 'EYEQ.zip'
RESULTS_ROOT    = DRIVE_ROOT / 'results'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'

# ----- Local Colab disk (working) -----
LOCAL_ROOT   = Path('/content/data')
APTOS_DIR    = LOCAL_ROOT / 'aptos'
EYEQ_DIR     = LOCAL_ROOT / 'eyeq'
PRISTINE_DIR = LOCAL_ROOT / 'pristine'
DEGRADED_DIR = LOCAL_ROOT / 'degraded'
ENHANCED_DIR = LOCAL_ROOT / 'enhanced'

PHASE_DIRS = {
    'phase1_data_engineering':   RESULTS_ROOT / 'phase1_data_engineering',
    'phase2_model_benchmarking': RESULTS_ROOT / 'phase2_model_benchmarking',
    'phase3_xai_benchmark':      RESULTS_ROOT / 'phase3_xai_benchmark',
    'phase4_genai_enhancement':  RESULTS_ROOT / 'phase4_genai_enhancement',
    'phase5_quality_ensemble':   RESULTS_ROOT / 'phase5_quality_ensemble',
}
PHASE_SUBDIRS = ('metrics', 'plots', 'samples', 'logs')

for p in [LOCAL_ROOT, APTOS_DIR, EYEQ_DIR, PRISTINE_DIR, DEGRADED_DIR, ENHANCED_DIR,
          RESULTS_ROOT, CHECKPOINTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

for phase_dir in PHASE_DIRS.values():
    for sub in PHASE_SUBDIRS:
        (phase_dir / sub).mkdir(parents=True, exist_ok=True)

print('Drive results tree:')
for k, v in PHASE_DIRS.items():
    print(f'  {k:35s} -> {v}')

## 4. Extract datasets from Drive to local disk

Unzipping to `/content/` is **much faster** than reading individual images from Drive during training. Re-run this cell at the start of every Colab session.

In [ ]:
import zipfile, shutil

def extract_zip(src_zip: Path, dest_dir: Path, force: bool = False):
    if not src_zip.exists():
        raise FileNotFoundError(f'Missing zip: {src_zip}')
    if any(dest_dir.iterdir()) and not force:
        print(f'[skip] {dest_dir} not empty')
        return
    if force and dest_dir.exists():
        shutil.rmtree(dest_dir); dest_dir.mkdir(parents=True)
    print(f'[extract] {src_zip.name} -> {dest_dir}')
    with zipfile.ZipFile(src_zip) as zf:
        zf.extractall(dest_dir)
    # Handle the case where the outer Thesis archive contains *.zip inside
    inner_zips = list(dest_dir.rglob('*.zip'))
    for iz in inner_zips:
        print(f'  inner zip detected: {iz.name}')
        with zipfile.ZipFile(iz) as zf:
            zf.extractall(iz.parent)
        iz.unlink()

extract_zip(APTOS_ZIP, APTOS_DIR)
extract_zip(EYEQ_ZIP,  EYEQ_DIR)

## 5. Inspect what was extracted

Run this and confirm you can see CSV(s) and image folders. If the layout differs from what's expected, edit the paths inside the next cell once and the phase notebooks will inherit them.

In [ ]:
def show_tree(root: Path, max_depth: int = 3):
    root = Path(root)
    for p in sorted(root.rglob('*')):
        depth = len(p.relative_to(root).parts)
        if depth > max_depth: continue
        kind = '/' if p.is_dir() else ''
        print(f'{"  "*(depth-1)}{p.name}{kind}')

print('=== APTOS ==='); show_tree(APTOS_DIR, max_depth=2)
print('\n=== EYEQ  ==='); show_tree(EYEQ_DIR,  max_depth=2)

## 6. Sanity check — GPU + library versions

In [ ]:
import torch, torchvision, timm, sys
print('Python      :', sys.version.split()[0])
print('PyTorch     :', torch.__version__)
print('Torchvision :', torchvision.__version__)
print('timm        :', timm.__version__)
print('CUDA avail  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU         :', torch.cuda.get_device_name(0))

---

**Done.** Now run the notebooks in order:

1. `01_phase1_data_engineering.ipynb` — quality filter + synthetic degradations
2. `02_phase2_model_benchmarking.ipynb` — train ResNet-50 / EfficientNet-B3 / ViT-Base, stress-test
3. `03_phase3_xai_benchmark.ipynb` — Grad-CAM / SHAP / Attention + faithfulness/stability metrics
4. `04_phase4_genai_enhancement.ipynb` — CLAHE + GenAI restoration, accuracy & XAI recovery
5. `05_phase5_quality_aware_ensemble.ipynb` — final routing pipeline + clinical trust score